**Data Source:** Airbnb New York City 2019 Open Data (Kaggle). The only dataset allowed in this task.

**Size:** 48,895 rows × 16 columns

| Column | Meaning | Type |
|---|---|---|
| id | Listing ID | Integer |
| name | Listing title | String |
| host_id | Host ID | Integer |
| host_name | Host name | String |
| neighbourhood_group | Borough | String |
| neighbourhood | Neighbourhood | String |
| latitude / longitude | Coordinates | Float |
| room_type | Room type | String |
| price | Nightly price (USD) | Integer |
| minimum_nights | Minimum nights | Integer |
| number_of_reviews | Total reviews | Integer |
| last_review | Date of last review | Date |
| reviews_per_month | Avg. reviews per month | Float |
| calculated_host_listings_count | Host's total listings | Integer |
| availability_365 | Available days per year | Integer |

**Note:** No rating column exists in the dataset; review counts will be used as a popularity proxy.

**Questions to Answer:**
1. What is the most common listing type?
2. Which areas are the most active?
3. How do prices differ across areas?
4. Is there a relationship between price, number of reviews, and review frequency?
5. Which factors appear to affect listing prices?
6. Which listings or areas show the strongest performance?
7. How do multi-listing (commercial) hosts affect the market?
8. What clear patterns or trends exist in the data over time?

In [44]:
import numpy as np 
import pandas as pd 
import matplotlib.pyplot as plt 
import seaborn as sns           
import plotly.express as px 
import plotly.graph_objects as go
from plotly.subplots import make_subplots
!pip install -U kaleido

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

sns.set_theme(style= 'whitegrid')
pd.set_option('display.max_columns', None)

/kaggle/input/datasets/mervebzn/nyc-airbnb/AB_NYC_2019.csv
/kaggle/input/datasets/mervebzn/nyc-airbnb/New_York_City_.png


# Data Understanding

In [45]:
path = '/kaggle/input/datasets/mervebzn/nyc-airbnb/AB_NYC_2019.csv'

df= pd.read_csv(path)

df.head(5)


,id,name,host_id,host_name,neighbourhood_group,neighbourhood,latitude,longitude,room_type,price,minimum_nights,number_of_reviews,last_review,reviews_per_month,calculated_host_listings_count,availability_365
0,2539,Clean & quiet apt home by the park,2787,John,Brooklyn,Kensington,40.64749,-73.97237,Private room,149,1,9,2018-10-19,0.21,6,365
1,2595,Skylit Midtown Castle,2845,Jennifer,Manhattan,Midtown,40.75362,-73.98377,Entire home/apt,225,1,45,2019-05-21,0.38,2,355
2,3647,THE VILLAGE OF HARLEM....NEW YORK !,4632,Elisabeth,Manhattan,Harlem,40.80902,-73.94190,Private room,150,3,0,NaN,NaN,1,365
3,3831,Cozy Entire Floor of Brownstone,4869,LisaRoxanne,Brooklyn,Clinton Hill,40.68514,-73.95976,Entire home/apt,89,1,270,2019-07-05,4.64,1,194
4,5022,Entire Apt: Spacious Studio/Loft by central park,7192,Laura,Manhattan,East Harlem,40.79851,-73.94399,Entire home/apt,80,10,9,2018-11-19,0.10,1,0


In [46]:
print('Rows : ', df.shape[0])
print('Columns : ', df.shape[1])

df.info()

Rows :  48895
Columns :  16
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 48895 entries, 0 to 48894
Data columns (total 16 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   id                              48895 non-null  int64  
 1   name                            48879 non-null  object 
 2   host_id                         48895 non-null  int64  
 3   host_name                       48874 non-null  object 
 4   neighbourhood_group             48895 non-null  object 
 5   neighbourhood                   48895 non-null  object 
 6   latitude                        48895 non-null  float64
 7   longitude                       48895 non-null  float64
 8   room_type                       48895 non-null  object 
 9   price                           48895 non-null  int64  
 10  minimum_nights                  48895 non-null  int64  
 11  number_of_reviews               48895 non-null  int64  
 12  last

In [47]:
df.describe().T

,count,mean,std,min,25%,50%,75%,max
id,48895.0,1.901714e+07,1.098311e+07,2539.00000,9.471945e+06,1.967728e+07,2.915218e+07,3.648724e+07
host_id,48895.0,6.762001e+07,7.861097e+07,2438.00000,7.822033e+06,3.079382e+07,1.074344e+08,2.743213e+08
latitude,48895.0,4.072895e+01,5.453008e-02,40.49979,4.069010e+01,4.072307e+01,4.076311e+01,4.091306e+01
longitude,48895.0,-7.395217e+01,4.615674e-02,-74.24442,-7.398307e+01,-7.395568e+01,-7.393627e+01,-7.371299e+01
price,48895.0,1.527207e+02,2.401542e+02,0.00000,6.900000e+01,1.060000e+02,1.750000e+02,1.000000e+04
minimum_nights,48895.0,7.029962e+00,2.051055e+01,1.00000,1.000000e+00,3.000000e+00,5.000000e+00,1.250000e+03
number_of_reviews,48895.0,2.327447e+01,4.455058e+01,0.00000,1.000000e+00,5.000000e+00,2.400000e+01,6.290000e+02
reviews_per_month,38843.0,1.373221e+00,1.680442e+00,0.01000,1.900000e-01,7.200000e-01,2.020000e+00,5.850000e+01
calculated_host_listings_count,48895.0,7.143982e+00,3.295252e+01,1.00000,1.000000e+00,1.000000e+00,2.000000e+00,3.270000e+02
availability_365,48895.0,1.127813e+02,1.316223e+02,0.00000,0.000000e+00,4.500000e+01,2.270000e+02,3.650000e+02


In [48]:
df.describe(include = 'object').T

,count,unique,top,freq
name,48879,47905,Hillside Hotel,18
host_name,48874,11452,Michael,417
neighbourhood_group,48895,5,Manhattan,21661
neighbourhood,48895,221,Williamsburg,3920
room_type,48895,3,Entire home/apt,25409
last_review,38843,1764,2019-06-23,1413


In [49]:
# Missing Values
missing = pd.DataFrame({
    'missing_count': df.isnull().sum(),
    'missing_pct' : (df.isnull().mean() * 100).round(2)
})

print(missing)
missing[missing["missing_count"] > 0]

                                missing_count  missing_pct
id                                          0         0.00
name                                       16         0.03
host_id                                     0         0.00
host_name                                  21         0.04
neighbourhood_group                         0         0.00
neighbourhood                               0         0.00
latitude                                    0         0.00
longitude                                   0         0.00
room_type                                   0         0.00
price                                       0         0.00
minimum_nights                              0         0.00
number_of_reviews                           0         0.00
last_review                             10052        20.56
reviews_per_month                       10052        20.56
calculated_host_listings_count              0         0.00
availability_365                            0         0.

,missing_count,missing_pct
name,16,0.03
host_name,21,0.04
last_review,10052,20.56
reviews_per_month,10052,20.56


In [50]:
# Duplicate rows
print('Duplicated rows:', df.duplicated().sum())

print('Duplicated ids:', df['id'].duplicated().sum())

Duplicated rows: 0
Duplicated ids: 0


In [51]:
# Unique 
print(df['neighbourhood_group'].value_counts(), '\n', 'Unique neighbourhoods: ', df['neighbourhood'].nunique())

print('\n', df['room_type'].value_counts(), '\n', 'Unique hosts: ', df['host_id'].nunique())

neighbourhood_group
Manhattan        21661
Brooklyn         20104
Queens            5666
Bronx             1091
Staten Island      373
Name: count, dtype: int64 
 Unique neighbourhoods:  221

 room_type
Entire home/apt    25409
Private room       22326
Shared room         1160
Name: count, dtype: int64 
 Unique hosts:  37457


In [52]:
# Zero Proces & Extreme Values
print(' Listings with price = 0 :', (df['price']==0).sum())
print('Listings with price > 1000 : ', (df['price']>1000).sum())
print('Listings with minimum_nights > 365 : ', (df['minimum_nights']> 365).sum())
print('Max price : ', df['price'].max())
print('Max minimum_nights : ', df['minimum_nights'].max())

 Listings with price = 0 : 11
Listings with price > 1000 :  239
Listings with minimum_nights > 365 :  14
Max price :  10000
Max minimum_nights :  1250


# Data Cleaning

In [53]:
df_clean = df.copy()
rows_start =  len(df_clean)

print('Starting rows:', rows_start)

Starting rows: 48895


In [54]:
# Missing Values
df_clean['name'] = df_clean['name'].fillna('Unknown')
df_clean['host_name'] = df_clean['host_name'].fillna('Unknown')

In [55]:
check = df_clean[df_clean['reviews_per_month'].isnull()]['number_of_reviews'].eq(0).all()

print('All missing have 0 reviews?:', check)

All missing have 0 reviews?: True


In [56]:
df_clean['reviews_per_month'] = df_clean['reviews_per_month'].fillna(0)

In [57]:
print(df_clean.isnull().sum()[df_clean.isnull().sum() > 0])

last_review    10052
dtype: int64


In [58]:
# Last Review convert to Datetime
df_clean['last_review']= pd.to_datetime(df_clean['last_review'], errors='coerce')


In [59]:
for col in ['name', 'host_name', 'neighbourhood_group', 'neighbourhood', 'room_type']:
    df_clean[col] = df_clean[col].str.strip()

In [60]:
# Convert categorical columns to 'category' dtype
for col in ['neighbourhood_group', 'room_type']:
    df_clean[col] = df_clean[col].astype('category')

In [61]:
print(df_clean.dtypes)
print('Date range:', df_clean['last_review'].min(), '->', df_clean["last_review"].max())

id                                         int64
name                                      object
host_id                                    int64
host_name                                 object
neighbourhood_group                     category
neighbourhood                             object
latitude                                 float64
longitude                                float64
room_type                               category
price                                      int64
minimum_nights                             int64
number_of_reviews                          int64
last_review                       datetime64[ns]
reviews_per_month                        float64
calculated_host_listings_count             int64
availability_365                           int64
dtype: object
Date range: 2011-03-28 00:00:00 -> 2019-07-08 00:00:00


In [62]:
# Invalid Records
n_zero = (df_clean['price']<= 0).sum()
df_clean = df_clean[df_clean['price']>0]
print('Dropped zero-price rows:', n_zero)

Dropped zero-price rows: 11


In [63]:
(df_clean['price']==0).sum()

np.int64(0)

In [64]:
# Unrealistic  minimum_nights
n_min = (df_clean['minimum_nights']>365).sum()
df_clean = df_clean[df_clean['minimum_nights'] <=365]

print('Dropped rows:', n_min)

Dropped rows: 14


In [65]:
# Outliers
Q1 = df_clean['price'].quantile(0.25)
Q3 = df_clean['price'].quantile(0.75)
IQR = Q3 - Q1
upper = Q3 + 1.5 * IQR

print(f'Q1={Q1}, Q3={Q3}, IQR={IQR}, upper bound={upper}')

print(df_clean["price"].quantile([0.5, 0.9, 0.95, 0.99]))

Q1=69.0, Q3=175.0, IQR=106.0, upper bound=334.0
0.50    106.0
0.90    269.0
0.95    355.0
0.99    799.0
Name: price, dtype: float64


In [66]:
# The IQR bound (~$334) is too strict for NYC and would remove luxury listings.
# So we FLAG instead of deleting, and use a $500 cap in the analysis.

df_clean['is_price_outlier'] = df_clean['price']>500
print(df_clean['is_price_outlier'].value_counts())
print('Outlier %:', round(df_clean['is_price_outlier'].mean() * 100, 2))

is_price_outlier
False    47826
True      1044
Name: count, dtype: int64
Outlier %: 2.14


## Feature engineering 

In [67]:
# Inactive listing
df_clean['is_inactive'] = df_clean['availability_365'] == 0

# Listing with no reviews
df_clean['has_reviews'] = df_clean['number_of_reviews'] > 0

# Host type: single listing vs multi-listing (commercial)
df_clean['host_type'] = np.where(
    df_clean['calculated_host_listings_count']>1, 'Multi-listing', 'Single-listing'
)

# Stay length category
df_clean['stay_type'] = pd.cut(
    df_clean['minimum_nights'],
    bins =[0, 3, 7, 30, 365],
    labels= ['Short (1-3)', 'Medium (4-7)', 'Long (8-30)', 'Monthly+ (31-365)']
)

# Price category (for readability in analysis)
df_clean['price_category'] = pd.cut(
    df_clean['price'],
    bins=[0, 75, 150, 250, 500, 10000],
    labels= ['Budget (<=75)', 'Mid (76-150)', 'Upper-mid (151-250)', 'Premium (251-500)', 'Luxury (>500)']
)

# Review year(for time trend)
df_clean['last_review_year'] = df_clean['last_review'].dt.year

# Log transform of price
df_clean['log_price']  = np.log1p(df_clean['price'])

In [68]:
# Unnecessary columns
df_clean= df_clean.drop(columns=['name', 'host_name'])

print('Remaining columns:', df_clean.columns.tolist())

Remaining columns: ['id', 'host_id', 'neighbourhood_group', 'neighbourhood', 'latitude', 'longitude', 'room_type', 'price', 'minimum_nights', 'number_of_reviews', 'last_review', 'reviews_per_month', 'calculated_host_listings_count', 'availability_365', 'is_price_outlier', 'is_inactive', 'has_reviews', 'host_type', 'stay_type', 'price_category', 'last_review_year', 'log_price']


In [69]:
# Comparison
rows_end = len(df_clean)
print('Start:', rows_start)
print('End:', rows_end)
print('Dropped:', rows_start - rows_end,
     f"({(rows_start - rows_end)/ rows_start * 100:.2f}%)")
print('Columns:', df_clean.shape[1])

Start: 48895
End: 48870
Dropped: 25 (0.05%)
Columns: 22


In [70]:
print('Remaining nulls:')
print(df_clean.isnull().sum()[df_clean.isnull().sum() > 0])

Remaining nulls:
last_review         10043
last_review_year    10043
dtype: int64


In [71]:
df_clean .to_csv('/kaggle/working/AB_NYC_2019_clean.csv', index= False)


# EDA

In [72]:
FIG_DIR = "/kaggle/working/figures"
os.makedirs(FIG_DIR, exist_ok=True)

COLORS = {
    "violet": "#7C3AED",
    "pink":   "#EC4899",
    "orange": "#F97316",
    "yellow": "#FACC15",
    "teal":   "#14B8A6",
    "blue":   "#3B82F6",
}
PALETTE = list(COLORS.values())

BORO_COLORS = {
    "Manhattan":     COLORS["violet"],
    "Brooklyn":      COLORS["pink"],
    "Queens":        COLORS["orange"],
    "Bronx":         COLORS["teal"],
    "Staten Island": COLORS["blue"],
}

ROOM_COLORS = {
    "Entire home/apt": COLORS["violet"],
    "Private room":    COLORS["pink"],
    "Shared room":     COLORS["orange"],
}

px.defaults.template = "plotly_white"
px.defaults.color_discrete_sequence = PALETTE

def save_fig(fig, name):
    # Saves the figure as both HTML (interactive) and PNG.
    fig.write_html(f"{FIG_DIR}/{name}.html")
    try:
        fig.write_image(f"{FIG_DIR}/{name}.png", scale=2)
        print(f"Saved: {name}.html + {name}.png")
    except Exception as e:
        print(f"Saved: {name}.html ( no PNG: {type(e).__name__})")

In [73]:
# raw price and log price
fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=('Raw price (USD)', 'log(1 + price)')
)

fig.add_trace(
    go.Histogram(x= df_clean['price'], nbinsx=100, marker_color=COLORS['pink'], name='price'),
    row=1, col=1
)
fig.add_trace(
    go.Histogram(x=df_clean["log_price"], nbinsx=60, marker_color=COLORS["violet"], name="log_price"),
    row=1, col=2
)

fig.update_layout(
    title='Price Distribution: Why Skewness Matters',
    showlegend=False, bargap=0.05, height=450
)

fig.update_xaxes(title_text='Price', row=1, col=1)
fig.update_xaxes(title_text="log(1 + price)", row=1, col=2)
fig.update_yaxes(title_text="Listings", row=1, col=1)

fig.show()
save_fig(fig, '01_price_distribution')

Saved: 01_price_distribution.html ( no PNG: ValueError)


**Price distribution**
- Price is right-skewed; the log transform makes it symmetric.

In [74]:
# Most common listing type
room_counts = df_clean['room_type'].value_counts().reset_index()
room_counts.columns = ['room_type', 'listings']
print(room_counts)

fig = px.pie(
    room_counts, names='room_type', values='listings',
    hole=0.55,
    color='room_type', color_discrete_map=ROOM_COLORS,
    title='Room Type Distribution'
)

fig.update_traces(textinfo='percent+label', textfont_size=14)
fig.update_layout(showlegend= False, height=450)

fig.show()
save_fig(fig, "02_room_type_donut")

         room_type  listings
0  Entire home/apt     25398
1     Private room     22315
2      Shared room      1157


Saved: 02_room_type_donut.html ( no PNG: ValueError)


**Room type**
- Entire home/apt %52, Private room %45,7, Shared room %2,4. / Two room types make up 98% of supply.
- Listing count shows supply, not demand.

In [75]:
# Most active neighbourhoods
top_nb = (
    df_clean.groupby(['neighbourhood', 'neighbourhood_group'], observed= True)
    .size().reset_index(name='listings')
    .sort_values('listings', ascending= False)
    .head(15)
)

fig = px.bar(
    top_nb.sort_values('listings'),
    x='listings', y='neighbourhood',
    color='neighbourhood_group', color_discrete_map=BORO_COLORS,
    orientation='h', text='listings',
    title='Top 15 Neighbourhoods by Listings',
    labels= {'listings': 'Listings', 'neighbourhood': '', 'neighbourhood_group': 'Borough'}
)

fig.update_traces(textposition='outside')
fig.update_layout(height= 600)

fig.show()
save_fig(fig, "03_top_neighbourhoods")

Saved: 03_top_neighbourhoods.html ( no PNG: ValueError)


**Most active areas**
- Top neighbourhoods :Williamsburg (3.917), Bedford-Stuyvesant (3.709), Harlem (2.656)
- Manhattan and Brooklyn make up ~85% of the market.
- Reviews per listing: Manhattan lowest.

In [76]:
# Borough summary and share comparison
# Summary table per borough
boro = (
    df_clean.groupby('neighbourhood_group', observed= True)
    .agg(
        listings=('id','count'),
        total_reviews=('number_of_reviews', 'sum'),
        median_price= ('price', 'median'),
        mean_price=('price', 'mean'),
    )
    .reset_index()
)
# Share of listings and reviews (%)
boro['listings_share_pct'] = (boro['listings']/ boro['listings'].sum() * 100).round(1)
boro['reviews_share_pct'] = (boro['total_reviews']/ boro['total_reviews'].sum() * 100).round(1)
boro['mean_price'] = boro['mean_price'].round(1)
display(boro)

share_long = boro.melt(
    id_vars= 'neighbourhood_group',
    value_vars=['listings_share_pct', 'reviews_share_pct'],
    var_name='metric',
    value_name='share_pct'
)

fig= px.bar(
    share_long, x='neighbourhood_group', y='share_pct',
    color='metric', barmode='group',text='share_pct',
    color_discrete_map={
        'Listing share' : COLORS['orange'],
        'Review share' : COLORS['yellow'],
    },
    title='Borough Share of Listings and Reviews (%)',
    labels = {
        'share_pct': 'Share (%)',
        'neighbourhood_group' : 'Borough',
        'metric' : ''
    }
)

fig.update_traces(textposition='outside')
fig.update_layout(height=480)

fig.show()
save_fig(fig, "04_borough_share")

,neighbourhood_group,listings,total_reviews,median_price,mean_price,listings_share_pct,reviews_share_pct
0,Bronx,1090,28316,65.0,87.6,2.2,2.5
1,Brooklyn,20089,486212,90.0,124.5,41.1,42.7
2,Manhattan,21654,454565,150.0,196.9,44.3,40.0
3,Queens,5664,156920,75.0,99.5,11.6,13.8
4,Staten Island,373,11541,75.0,114.8,0.8,1.0


Saved: 04_borough_share.html ( no PNG: ValueError)


In [77]:
# Price by borough
df_price= df_clean[df_clean['price']<=500]

order=(
    df_price.groupby('neighbourhood_group',observed=True)['price']
    .median().sort_values(ascending= False).index.tolist()
)

fig = px.box(
    df_price, x='neighbourhood_group', y='price',
    color='neighbourhood_group', color_discrete_map=BORO_COLORS,
    category_orders={'neighbourhood_group': order},
    points=False,
    title='Price Distribution by Borough (≤ $500)',
    labels={'price': 'Nightly price (USD)', 'neighbourhood_group': 'Borough'}
)

fig.update_layout(showlegend=False, height=500)

fig.show()
save_fig(fig, "05_price_by_borough_box")
print(df_price.groupby("neighbourhood_group", observed=True)["price"]
      .describe()[["count", "mean", "25%", "50%", "75%"]].round(1))

Saved: 05_price_by_borough_box.html ( no PNG: ValueError)
                       count   mean   25%    50%    75%
neighbourhood_group                                    
Bronx                 1083.0   81.8  45.0   65.0   99.0
Brooklyn             19860.0  112.7  60.0   90.0  149.0
Manhattan            20881.0  163.2  92.0  145.0  200.0
Queens                5635.0   92.8  50.0   75.0  110.0
Staten Island          367.0   91.1  50.0   75.0  105.5


**Price by borough**
- Median price ranking : Manhattan $145 > Brooklyn $90 > Queens/Staten Island $75 > Bronx $65.
- 75% of listings are $200 or less everywhere.

In [78]:
# Price vs number of reviews
fig = px.box(
    df_clean[df_clean['price'] <= 500],
    x='price_category',
    y='number_of_reviews',
    color='neighbourhood_group',
    color_discrete_map=BORO_COLORS,
    log_y=True,
    title='Price category and number of reviews',
    labels={
        'price_category': 'Fiyat Kategorisi',
        'number_of_reviews': 'Yorum Sayısı (log)',
        'neighbourhood_group': 'Bölge'
    }
)

fig.update_layout(height=600, boxmode='group')

fig.show()
save_fig(fig, '06_price_cat_vs_reviews_box')

Saved: 06_price_cat_vs_reviews_box.html ( no PNG: ValueError)


**Price category and number of reviews**
- Median reviews (reviewed listings only): Budget 9, Mid 10, Upper-mid 9, Premium 8, Luxury 6.
- The first three groups are nearly identical; the clear drop appears in the Luxury group.
- Share of listings with at least one review: 82% in Mid vs 60% in Luxury.

In [79]:
# Review frequency by price category
price_rpm = (
    df_clean[df_clean['has_reviews']]
    .groupby('price_category', observed= True)['reviews_per_month']
    .median().round(2).reset_index()
)
print(price_rpm)

fig= px.bar(
    price_rpm, x='price_category', y='reviews_per_month',
    text='reviews_per_month', color='price_category',
    color_discrete_sequence=PALETTE,
    title='Median Reviews per Month by Price Category',
    labels={
        'price_category': 'Price category',
        'reviews_per_month' : 'Reviews per month (median)'
    }
)

fig.update_traces(textposition='outside')
fig.update_layout(showlegend=False, height=600)

fig.show()
save_fig(fig, "07_reviews_by_price_category")

        price_category  reviews_per_month
0        Budget (<=75)               0.76
1         Mid (76-150)               0.69
2  Upper-mid (151-250)               0.65
3    Premium (251-500)               0.85
4        Luxury (>500)               0.59


Saved: 07_reviews_by_price_category.html ( no PNG: ValueError)


**Review frequency by price category**
- Median reviews per month: Budget 0.76, Mid 0.69, Upper-mid 0.65, Premium 0.85, Luxury 0.59.
- The Premium value is slightly higher than its neighbours; the gap is small (about 0.2 reviews per month) and has no clear explanation in the data.

In [80]:
# Correlation heatmap
num_cols = ['price', 'minimum_nights', 'number_of_reviews', 'reviews_per_month', 'calculated_host_listings_count', 'availability_365']

corr = df_clean[num_cols].corr(method='spearman').round(2)

fig = px.imshow(
    corr, text_auto = True, aspect='auto',
    color_continuous_scale=[COLORS['blue'], "#FFFFFF", COLORS['pink']],
    zmin=1, zmax=1,
    title= 'Spearman Correlation Matrix'
)
fig.update_layout(height=550)

fig.show()
save_fig(fig, "08_correlation_heatmap")

Saved: 08_correlation_heatmap.html ( no PNG: ValueError)


**Correlations (Spearman)**
- Price has weak correlations with all numeric variables (|r| ≤ 0.11).
- number_of_reviews and reviews_per_month: r = 0.85, expected because one is derived from the other.
- minimum_nights and reviews_per_month: r = -0.25. Median reviews per month fall from 1.00 (1-3 nights) to 0.40 (4-7), 0.26 (8-30) and 0.23 (31+).
- availability_365 and calculated_host_listings_count: r = 0.41; hosts with more listings keep their calendars more open.

In [81]:
# Borough x room type price heatmap
pivot = df_clean.pivot_table(
    index= 'neighbourhood_group', columns='room_type',
    values= 'price', aggfunc='median', observed=True
)
print(pivot)

fig = px.imshow(
    pivot, text_auto = True, aspect='auto',
    color_continuous_scale=[COLORS['yellow'], COLORS['orange'], COLORS['pink'], COLORS['violet']],
    title= 'Median Price: Borough x Room Type',
    labels={'color': 'USD'}
)
fig.update_layout(height = 450)

fig.show()
save_fig(fig, "09_price_heatmap_borough_room")

room_type            Entire home/apt  Private room  Shared room
neighbourhood_group                                            
Bronx                          100.0          54.0         40.0
Brooklyn                       145.0          65.0         36.0
Manhattan                      191.0          90.0         69.0
Queens                         120.0          60.0         37.0
Staten Island                  100.0          50.0         30.0


Saved: 09_price_heatmap_borough_room.html ( no PNG: ValueError)


**Median price by borough and room type**
- Manhattan is the most expensive in all three room types (entire home $191, private room $90, shared room $69).
- Within the same room type, prices still differ by borough (entire home: Manhattan $191, Brooklyn $145, Queens $120, Bronx and Staten Island $100).
- The Manhattan premium is therefore not explained by room-type mix alone.

In [82]:
# Real demand by room type
room_perf = (
    df_clean[df_clean['has_reviews']]
    .groupby('room_type', observed=True)
    .agg(
        median_price=('price', 'median'),
        mean_rpm=('reviews_per_month', 'mean'))
    .round(2).reset_index()
)

print(room_perf)

fig = make_subplots(
    rows=1, cols=2,
    subplot_titles=(' Median price', 'Avg. reviews per month')
)
fig.add_trace(go.Bar(
    x=room_perf['room_type'], y=room_perf['median_price'],
    marker_color=COLORS['violet'], text=room_perf['median_price'], textposition='outside'
), row=1, col=1)

fig.add_trace(go.Bar(
    x=room_perf['room_type'], y=room_perf['mean_rpm'],
    marker_color=COLORS['teal'], text=room_perf['mean_rpm'], textposition='outside'
), row=1, col=2)

fig.update_layout(
    title='Room Type: Price and Demand',
    showlegend= False, height= 470
)

fig.show()
save_fig(fig, "10_room_type_price_demand")

         room_type  median_price  mean_rpm
0  Entire home/apt         151.0      1.31
1     Private room          70.0      1.44
2      Shared room          45.0      1.47


Saved: 10_room_type_price_demand.html ( no PNG: ValueError)


**Room type: price and demand**
- Median price: Entire home $151, Private room $70, Shared room $45.
- Average reviews per month: Entire home 1.31, Private room 1.44, Shared room 1.47.
- Cheaper room types receive reviews slightly more often despite far lower prices.

In [83]:
# Time trend
df_time = df_clean.dropna(subset=['last_review'])
monthly = (
    df_time.set_index('last_review')
    .resample('MS').size()
    .reset_index(name='listings')
)
monthly = monthly[monthly['last_review'] >='2017-01-01']

fig = px.line(
    monthly, x='last_review', y='listings', markers=True,
    title='Listings by Month of Last Review',
    labels={
        'last_review' : 'Month', 
        'listings' : 'Listings'
    }  
)

fig.update_traces(line_color=COLORS['violet'], marker_color=COLORS['pink'], line_width=3)
fig.update_layout(height=480)

fig.show()
save_fig(fig, "15_last_review_trend")

print(df_time['last_review'].dt.year.value_counts().sort_index())

Saved: 15_last_review_trend.html ( no PNG: ValueError)
last_review
2011        7
2012       25
2013       48
2014      198
2015     1393
2016     2706
2017     3204
2018     6044
2019    25202
Name: count, dtype: int64


**Time pattern (last review date)**
- last_review stores only the most recent review of each listing, so this chart shows recency of activity, not review growth.
- 25,202 of the 38,827 listings with a review date (65%) had their last review in 2019; 2018 accounts for 6,044 (16%).
- Monthly counts rise sharply in 2019: April 1,456, May 3,695, June 12,596.
- The dataset is a snapshot (latest date: 8 July 2019), so the recent peak reflects when the data was collected and should not be read as a seasonal peak.

In [84]:
rpm_boro = (
    df_clean[df_clean["has_reviews"]]
    .groupby("neighbourhood_group", observed=True)["reviews_per_month"]
    .median().round(2)
)
print(rpm_boro, "\n")

df_clean["host_group"] = pd.cut(
    df_clean["calculated_host_listings_count"],
    bins=[0, 1, 5, 20, 1000],
    labels=["1 listing", "2-5", "6-20", "21+"]
)
host_tbl = (
    df_clean.groupby("host_group", observed=True)
    .agg(listings=("id", "count"),
         hosts=("host_id", "nunique"),
         median_availability=("availability_365", "median"),
         pct_inactive=("is_inactive", "mean"))
    .reset_index()
)
host_tbl["pct_inactive"] = (host_tbl["pct_inactive"] * 100).round(1)
host_tbl["listing_share_pct"] = (host_tbl["listings"] / host_tbl["listings"].sum() * 100).round(1)
host_tbl["host_share_pct"] = (host_tbl["hosts"] / host_tbl["hosts"].sum() * 100).round(1)
display(host_tbl)


print("Inactive share (%):", round(df_clean["is_inactive"].mean() * 100, 1))

neighbourhood_group
Bronx            1.38
Brooklyn         0.66
Manhattan        0.61
Queens           1.21
Staten Island    1.41
Name: reviews_per_month, dtype: float64 



,host_group,listings,hosts,median_availability,pct_inactive,listing_share_pct,host_share_pct
0,1 listing,32288,32288,6.0,46.2,66.1,86.2
1,2-5,11792,4809,116.0,20.6,24.1,12.8
2,6-20,2585,307,283.0,5.7,5.3,0.8
3,21+,2205,38,319.0,2.2,4.5,0.1


Inactive share (%): 35.9
